# Credit Risk Prediction Using Decisiom Trees and Random Forests

## Objective: Predict whether a customer will default on credit and compare baseline and optimized Decision Tree and Random Forest models.

# STEP 1 - DOWNLOAD THE DATASET

#### Import Pandas module
#### Load the read data in a datafram called df

In [2]:
import pandas as pd

file_path = "datasets/credit-risk-dataset/credit_risk_dataset.csv"

df = pd.read_csv(file_path)

# STEP 2 - INSPECTING THE FIRST ROWS

#### df.head() gives me the first 5 rows

In [3]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


# STEP 3 - IDENTIFYING THE TARGET

#### The target Y will be Loan Status because that is what we are trying to predict, whether the customer will default or not

# STEP 4 - CHECK THE DATASET STRUCTURE

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


#### The two columns with missing values are person_emp_length and and loan_int_rate. I can tell because the count is not the total of 32581 like the others

In [5]:
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

#### df.isnull().sum checks every column that has a null value, tallies them and gives me the count

In [6]:
person_emp_length_percentage = (df["person_emp_length"].isnull().sum() / len(df)) * 100
print(person_emp_length_percentage)

2.7469997851508547


In [7]:
loan_int_rate = (df["loan_int_rate"].isnull().sum() / len(df)) * 100
print(loan_int_rate)

9.563856235229121


#### person_emp_length_percentage and loan_int_rate let us see what percentage of data is missing in each column where null values are reported. 
#### If they are insignificant. we could do away with the data. But if they are significant we may need another way to mak eup fo rthe loss data. 

#### person_emp_length → 2.75% 
#### missing loan_int_rate → 9.56% 

#### Before we decide how to fill them, we should understand the values in those two numeric columns.

In [8]:
df['person_emp_length'].describe()

count    31686.000000
mean         4.789686
std          4.142630
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max        123.000000
Name: person_emp_length, dtype: float64

In [9]:
df['loan_int_rate'].describe()

count    29465.000000
mean        11.011695
std          3.240459
min          5.420000
25%          7.900000
50%         10.990000
75%         13.470000
max         23.220000
Name: loan_int_rate, dtype: float64

We use <b>describe</b> to analyze the column. <b>Decsribe</b> tells me the mean, median, count, max lows and any other weird data I can catch<br>
In person_emp_length, the max years of 1234 is unreasonable. Thehe value 123 years of employment is almost certainly suspicious and could distort the model if left untreated.<br>
In that case we can check the person's age to see. And we can see the person who has worked for 123 is 22 years. That is quite impossible.<br>
One check we could apply is to check that someone's age is greater than their employment years <br?

In [10]:
df[df['person_emp_length'] > df['person_age']]

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4


### Data Quality Observation: Employment Length

The comparison identified **2 rows** where employment length is greater than the customer's age:

- **Row 0:** age = 22, employment length = 123
- **Row 210:** age = 21, employment length = 123

An employment length of `123` years is not realistic, so it should be treated as an <span style="color:red"><b>invalid value</b></span> in `person_emp_length`.

### Decision

We will **keep both customer records** because their other information may still be useful. Instead, we will:

1. Convert `123` to a missing value.
2. Handle it together with the other missing employment-length values.
3. Fill the missing values using the **median** employment length.

The median is preferred because the extreme value `123` pulls the **mean** upward, while the median is more resistant to outliers. This gives us a more realistic replacement value.

In [11]:
import numpy as np

df["person_emp_length"] = df["person_emp_length"].replace(123, np.nan)

In [12]:
df["person_emp_length"].isnull().sum()

np.int64(897)

### Missing-Value Correction

At **Cell 7**, `person_emp_length` had **895 null values**.

After replacing the two unrealistic `123` values with `NaN`, the number of null values increased to **897**:

- **895 original null values**
- **+ 2 invalid employment-length values**
- **= 897 total null values**

This confirms that the invalid values were successfully identified and converted to missing values. The `person_emp_length` column is now ready for **median imputation**.

<span style="color:red"><b>Conclusion:</b> The data has been cleaned without deleting the customer records.</span>

In [13]:
df['person_emp_length'].median()

np.float64(4.0)

In [17]:
median = df['person_emp_length'].median()
df["person_emp_length"] = df["person_emp_length"].fillna(median)
df["person_emp_length"].isnull().sum()

np.int64(0)

### Employment-Length Imputation Summary

- The cleaned `person_emp_length` column has a **median of 4 years**.
- Pandas automatically ignores `NaN` values when calculating the median.
- We used **`fillna()`** to replace the missing employment lengths with the median value.
- The result is **0 null values**, confirming that all missing employment lengths were successfully filled.

<span style="color:red"><b>Conclusion:</b> The `person_emp_length` column is now complete and ready for further analysis.</span>

In [18]:
median = df['loan_int_rate'].median()
df["loan_int_rate"] = df["loan_int_rate"].fillna(median)
df["loan_int_rate"].isnull().sum()

np.int64(0)

### Loan-Interest-Rate Imputation Summary

- The median `loan_int_rate` is **10.99%**.
- Pandas automatically ignores `NaN` values when calculating the median.
- We used **`fillna()`** to replace missing interest rates with the median value.
- The result is **0 null values**, confirming that all missing interest rates were successfully filled.

<span style="color:red"><b>Conclusion:</b> The `loan_int_rate` column is now complete and ready for further analysis.</span>

In [19]:
df.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

### We have fixed all our NULLs

# STEP 5 - ENCODING CATEGORICAL FEATURES

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           32581 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               32581 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


### CHECK THE VALUES IN EACH CATEGORY
Our categories:<br>
person_home_ownership<br>
loan_intent<br>
loan_grade<br>
cb_person_default_on_file<br>

In [21]:
df["person_home_ownership"].unique() 

<StringArray>
['RENT', 'OWN', 'MORTGAGE', 'OTHER']
Length: 4, dtype: str

In [22]:
df["loan_intent"].unique() 

<StringArray>
[         'PERSONAL',         'EDUCATION',           'MEDICAL',
           'VENTURE',   'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str

In [23]:
df["loan_grade"].unique() 

<StringArray>
['D', 'B', 'C', 'A', 'E', 'F', 'G']
Length: 7, dtype: str

In [24]:
df["cb_person_default_on_file"].unique() 

<StringArray>
['Y', 'N']
Length: 2, dtype: str

### Categorical-Feature Encoding Summary

We inspected the unique values in each categorical column to understand how the data should be prepared for machine learning. Scikit-learn's **Decision Tree** and **Random Forest** models require numeric input, so text categories such as `RENT` and `OWN` must be encoded.

Simple numeric labels are not suitable for unordered categories because the model might incorrectly interpret the numbers as having a meaningful ranking. For example, coding `RENT = 0`, `OWN = 1`, and `MORTGAGE = 2` could suggest that one category is greater than another.

However, `loan_grade` has a natural order from **A** (lower risk) to **G** (higher risk), so it can use ordinal encoding.

### Encoding Plan

| Feature | Encoding | Reason |
|---|---|---|
| `loan_grade` | Ordinal mapping: A = 1 through G = 7 | The grades have a meaningful order. |
| `cb_person_default_on_file` | Binary mapping: N = 0, Y = 1 | The feature has two possible values. |
| `person_home_ownership` | One-hot encoding | The categories have no natural order. |
| `loan_intent` | One-hot encoding | The categories have no natural order. |

<span style="color:red"><b>Conclusion:</b> Each categorical feature will use an encoding method that matches the meaning of its values, helping the models interpret the data correctly.</span>

### ENCODING

In [26]:
grade_mapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7
}

df["loan_grade"] = df["loan_grade"].map(grade_mapping)
df["loan_grade"].unique()

array([nan])

In [27]:
raw_df = pd.read_csv(file_path)
df["loan_grade"] = raw_df["loan_grade"]
df["loan_grade"].unique()


<StringArray>
['D', 'B', 'C', 'A', 'E', 'F', 'G']
Length: 7, dtype: str

In [29]:
df["loan_grade_encoded"] = df["loan_grade"].map(grade_mapping)
df[["loan_grade", "loan_grade_encoded"]].head()

,loan_grade,loan_grade_encoded
0,D,4
1,B,2
2,C,3
3,C,3
4,C,3


In [31]:
default_on_file_mapping = {
    "N": 0,
    "Y": 1
}

df["cb_person_default_encoded"] = df["cb_person_default_on_file"].map(default_on_file_mapping)
df[["cb_person_default_on_file", "cb_person_default_encoded"]].head()

,cb_person_default_on_file,cb_person_default_encoded
0,Y,1
1,N,0
2,N,0
3,N,0
4,Y,1


In [32]:
df = pd.get_dummies(
    df,
    columns=["person_home_ownership", "loan_intent"],
    drop_first=True,
    dtype=int
)

In [33]:
df.head()

,person_age,person_income,person_emp_length,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_grade_encoded,cb_person_default_encoded,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,59000,4.0,D,35000,16.02,1,0.59,Y,3,4,1,0,0,1,0,0,0,1,0
1,21,9600,5.0,B,1000,11.14,0,0.10,N,2,2,0,0,1,0,1,0,0,0,0
2,25,9600,1.0,C,5500,12.87,1,0.57,N,3,3,0,0,0,0,0,0,1,0,0
3,23,65500,4.0,C,35000,15.23,1,0.53,N,2,3,0,0,0,1,0,0,1,0,0
4,24,54400,8.0,C,35000,14.27,1,0.55,Y,4,3,1,0,0,1,0,0,1,0,0


In [34]:
df = df.drop(
    columns=[
        "loan_grade",
        "cb_person_default_on_file"
    ]
)

In [35]:
df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,loan_grade_encoded,cb_person_default_encoded,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,59000,4.0,35000,16.02,1,0.59,3,4,1,0,0,1,0,0,0,1,0
1,21,9600,5.0,1000,11.14,0,0.10,2,2,0,0,1,0,1,0,0,0,0
2,25,9600,1.0,5500,12.87,1,0.57,3,3,0,0,0,0,0,0,1,0,0
3,23,65500,4.0,35000,15.23,1,0.53,2,3,0,0,0,1,0,0,1,0,0
4,24,54400,8.0,35000,14.27,1,0.55,4,3,1,0,0,1,0,0,1,0,0


# Summary of Data Preparation

## 1. Encoding Categorical Features

Machine-learning models such as **Decision Trees** and **Random Forests** require numeric input, so the text categories had to be encoded carefully.

- `loan_grade`: **ordinal encoding**, because grades have a natural order from **A** (lower risk) to **G** (higher risk).
- `cb_person_default_on_file`: **binary encoding**, with `N = 0` and `Y = 1`.
- `person_home_ownership`: **one-hot encoding**, because categories such as `RENT`, `OWN`, and `MORTGAGE` have no natural order.
- `loan_intent`: **one-hot encoding**, because the loan-purpose categories are also unordered.

For one-hot encoding, `drop_first=True` was used to avoid unnecessary duplicate information. The original text columns were then removed after their numeric replacements were created.

<span style="color:red"><b>Overall conclusion:</b> The dataset has been inspected, cleaned, missing values have been imputed, and categorical features have been converted into meaningful numeric representations. The data is now prepared for building and evaluating the Decision Tree and Random Forest models.</span>

# STEP 6 - DATASET SPLITTING - SEPARATE X AND Y

In [38]:
y = df["loan_status"]

In [40]:
x = df.drop(
    columns=[
        "loan_status"
    ]
)
x.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'loan_grade_encoded', 'cb_person_default_encoded',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='str')

# Summary: Separating Features and Target

In this step, we separated the dataset into two parts for machine learning:

- **`X`** contains the predictor features used by the model.
- **`y`** contains the target variable, `loan_status`, which represents whether the customer defaulted on the loan.

We correctly excluded `loan_status` from `X` because it is the value we want the model to predict. Including it in the predictor features would allow the model to see the answer in advance and cause **data leakage**.

We then checked `X.columns` to verify that `loan_status` was removed successfully.

<span style="color:red"><b>Conclusion:</b> `X` contains only the input features, while `y` contains `loan_status`. The data is now correctly separated and ready for model training.</span>

# STEP 7 - Train/test split + stratification